# 23 - The constants the model eats, assembled and given provenance

**Purpose.** Assemble `results/model_constants.json` - the set of published constants `model.py`
consumes, each one carrying the provenance of the notebook that measured it.

This notebook measures nothing. It is the first file in `results/` derived from other files in
`results/` rather than from frames, and that is its whole reason to exist: `R(gain)` and `g(gain)`
are published as **sweeps** - `bias_sweep.csv` and `ptc_gain.csv` - and a sweep row carries no
provenance, so the model cannot legally eat one. Every stanza written here names the notebook that
took the frames, never this one.

**What it publishes.** `read_noise_counts` against gain over the characterised domain, from the
-10 C bias sweep. `system_gain` against gain, from the PTC. The scalars the model needs in one
place: `t_dead`, `F_sky` per plane, the dark-current bound, the pedestal fit, the HCG threshold,
the linearity ceiling, and `eta_comb` with what it was measured on attached.

**What it is not for.** It does not re-measure, re-fit, or improve any number it moves. If a value
here disagrees with its source file, the source file is right and this notebook is the bug. It
does not fill the gap at gain 252 in `linear_to_at_least` - that needs frames, not arithmetic.

**The known gap it publishes as a null.** `linear_to_at_least` stops at gain 200 and the
validation ladder sits at 252, so the star-colour ceiling there is published null with a reason
rather than extrapolated.

In [ ]:
import json
import pathlib
import sys

import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
MODEL_JSON = RESULTS / "model_constants.json"

# Every source, opened once.  Read as raw JSON rather than through
# `M.Constants`, because what is being moved here is the whole stanza -- note,
# uncertainty, the notebook that measured it -- and the loader deliberately
# hands back only the value.
src = {p.stem: json.loads(p.read_text()) for p in sorted(RESULTS.glob("*.json"))
       if p.name != MODEL_JSON.name}
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")

print(f"{len(src)} source files, {sum(len(v) for v in src.values())} constants between them")
print(f"bias_sweep.csv: {len(sweep)} rows")

## 1. Read noise against gain, from the bias sweep

Session 01 walked the gain axis in tens at the project offset, with a fine grid of twos through the
HCG threshold. That is the densest thing this project has on any axis, and it is the only source
for `R` at a gain nobody sat at - which includes 252, where the validation ladder was shot.

**Two filters, and both are rules rather than choices.** Only offset 15, because that is the offset
this project fixed on and the pedestal moves four counts per offset unit; and only gain 0-450,
per the gain-domain rule in `CLAUDE.md`. The rows above 450 stay in `bias_sweep.csv` where session
01 left them - descoping decides what is characterised, never what is deleted - they simply do not
become a constant.

`R_sd` is the pair-difference sigma over root two, which is the estimator this project uses
everywhere. At offset 15 it is identical to `R_at_offset` by construction, and the column is read
by name rather than assumed.

In [ ]:
DOMAIN = sweep[(sweep.offset == 15) & (sweep.gain <= M.GAIN_MAX)].sort_values("gain")
assert (DOMAIN.R_sd == DOMAIN.R_at_offset).all(), "R_at_offset should be R_sd at the fit offset"

read_noise = {str(int(r.gain)): round(float(r.R_sd), 6) for r in DOMAIN.itertuples()}
read_noise_err = {str(int(r.gain)): round(float(r.R_err), 6) for r in DOMAIN.itertuples()}
n_bias_frames = int(2 * DOMAIN.n_pairs.sum())

print(f"{len(read_noise)} gains, {DOMAIN.gain.min()} to {DOMAIN.gain.max()}, "
      f"{n_bias_frames} frames behind them")
print(f"R runs {DOMAIN.R_sd.min():.4f} counts at gain {int(DOMAIN.gain.iloc[0])} "
      f"to {DOMAIN.R_sd.max():.4f} at gain {int(DOMAIN.gain.iloc[-1])}, "
      f"worst standard error {DOMAIN.R_err.max():.6f}")

## 2. System gain, and a record decision this notebook does not make

`g` is published twice. `ptc_constants.json` holds session 02's fit, which is what MISSION's
constants table names. `state_repair_constants.json` holds session 04's restatement, made after it
found offset-state contamination in session 02's bias groups at five of the eight gains.

**Session 04 declined to supersede** - its own note says so in as many words, and calls it a record
decision rather than a notebook's. This notebook is not the place that decision gets made by
default, so it moves the PTC table and publishes the size of the disagreement beside it. The two
agree to better than 0.2% everywhere except gain 300 and gain 450.

Read noise in electrons is not published. It is `R_counts * g`, and the two tables behind it were
measured by different sessions on different frames; storing the product would create a third number
that could drift out of step with either. `model.read_noise_e` multiplies them at the point of use.

In [ ]:
g_ptc = src["ptc_constants"]["system_gain"]
g_restated = src["state_repair_constants"]["system_gain_restated"]
move = src["state_repair_constants"]["system_gain_move_pct"]["value"]

in_domain = {k: v for k, v in g_ptc["value"].items() if float(k) <= M.GAIN_MAX}
assert in_domain == g_ptc["value"], "the PTC sweep should already stop at the domain edge"

print("gain   PTC       restated   move")
for k in sorted(in_domain, key=float):
    print(f"{k:>4}  {in_domain[k]:8.5f}  {g_restated['value'][k]:8.5f}  {move[k]:+6.3f}%")
print(f"\nworst disagreement {max(abs(v) for v in move.values()):.3f}%, at gain "
      f"{max(move, key=lambda k: abs(move[k]))}")

## 3. The scalars, moved unchanged

Seven stanzas copied whole from the notebooks that measured them, provenance intact. Nothing is
recomputed; the only edit is a note saying which term of the model each one feeds, because a
constant file assembled for a consumer should say what the consumer does with it.

Three of them carry a warning in their own note and the warning travels with them:

- **`dark_current_bound` is a bound, not a value.** The implied rate disagrees in sign across
  exposures, which is what a dark current cannot do, and what limits it is the offset state rather
  than statistics. The model treats it as `D`, which is conservative in the only direction that
  matters: it can only make a sub look noisier than it is.
- **`F_sky` is an upper bound too**, and for an unrelated reason - unresolved nebulosity inside the
  darkest corner of an NGC 7000 field cannot be separated from sky. It is the right bound for this
  model, which wants the level sitting under the faint signal.
- **`eta_comb` was measured on bias stacks, not on registered lights.** It stalls against a
  fixed-pattern floor, which is why it falls to 0.536 by N=128. Its own note calls it an upper
  bound on the real loss. Contract 3 replaces it and until then it is the pessimistic end.

In [ ]:
MOVED = {
    "system_gain": ("ptc_constants", "feeds g(gain); model.g_at interpolates it log-linearly"),
    "pedestal_fit": ("bias_constants", "feeds the pedestal in the star-colour constraint"),
    "hcg_threshold_gain": ("bias_constants", "selects the branch of pedestal_fit"),
    "t_dead": ("sky_constants", "feeds N = T_night / (t + t_dead)"),
    "F_sky": ("sky_constants", "feeds F_sky, per CFA plane"),
    "dark_current_bound": ("dark_constants", "feeds D, at the fixed -10 C setpoint"),
    "eta_comb": ("dark_constants", "feeds eta_comb; model.eta_at interpolates it in log N"),
    "linear_to_at_least": ("linearity_constants", "feeds ceiling(gain) in the star-colour "
                           "constraint -- the proved-straight bound, never the clip"),
}

constants = {}
for name, (where, feeds) in MOVED.items():
    stanza = dict(src[where][name])
    stanza["note"] = f"[{feeds}]  moved unchanged from {where}.json.  " + stanza["note"]
    constants[name] = stanza
    print(f"{name:22s} <- {where}.json  ({stanza['notebook']}, {stanza['measured_on']})")

## 4. The gap at the validation gain, published as a null

The linearity ladder ran at gains 50, 100 and 200. The NGC 7000 exposure ladder - the dataset the
model's definition of done is written against - was shot at gain 252. So the star-colour half of
the model has no ceiling to compare against on the one dataset that matters.

**It is not interpolated.** `linear_to_at_least` is the last rung a ladder proved straight, and its
own resolution is one rung at 4% of `t_sat`; interpolating between two gains would invent a
guarantee that no frame supports. It costs a short bench block to fix - a linearity run at 252 -
and nothing else in the model is blocked by it.

The null is written into the file rather than left out, because a missing key is an oversight and a
null with a reason is a finding. `model.Constants` raises on it and quotes the note, which is the
behaviour that turns this gap into a message rather than a wrong answer.

In [ ]:
LADDER_GAIN = 252

constants["linear_to_at_ladder_gain"] = {
    "value": None, "unit": "ADC counts", "uncertainty": None,
    "source_frames": 0, "measured_on": None, "notebook": "23_model_constants.ipynb",
    "note": f"[feeds ceiling(gain) at the validation gain]  NOT MEASURED.  The linearity ladder "
            f"ran at gains {sorted(src['linearity_constants']['linear_to_at_least']['value'], key=float)} "
            f"and the NGC 7000 validation ladder was shot at gain {LADDER_GAIN}.  Deliberately not "
            f"interpolated: linear_to_at_least is the last rung a ladder PROVED straight, at a "
            f"resolution of one rung, and a value between two gains would be a guarantee no frame "
            f"supports.  Fixed by a short linearity block at gain {LADDER_GAIN}; nothing else in "
            f"the model waits on it",
}

constants["system_gain_restated_move_pct"] = dict(
    src["state_repair_constants"]["system_gain_move_pct"],
    note="[not consumed by the model]  how far session 04's restated g sits from the PTC table "
         "published above, per gain.  Carried so the size of an open record decision is visible "
         "in the file the model reads, without being a second table anything could eat by "
         "accident.  See state_repair_constants.json for the restated values themselves")

print(f"{sum(v['value'] is None for v in constants.values())} null with a reason, "
      f"{len(constants)} stanzas total")

## 5. The two constants this file adds, and what they are made of

`read_noise_counts` is the only stanza here that is assembled rather than moved, and its provenance
points at `03_bias_sweep.ipynb` - the notebook that took the frames - rather than at this one. That
is the rule this notebook exists to keep: the chain of custody runs back to the frames, and an
assembler that signed its own work would have broken it at the last step.

Its uncertainty is the per-gain standard error the sweep already carries, not a single figure,
because the read noise is a table and so is the error on it.

In [ ]:
constants["read_noise_counts"] = {
    "value": read_noise, "unit": "ADC counts", "uncertainty": read_noise_err,
    "source_frames": n_bias_frames,
    "measured_on": src["bias_constants"]["read_noise_at_hcg"]["measured_on"],
    "notebook": "03_bias_sweep.ipynb",
    "note": f"[feeds R(gain); model.read_noise_e multiplies it by g at the point of use]  "
            f"assembled by 23_model_constants.ipynb from bias_sweep.csv, which 03_bias_sweep.ipynb "
            f"wrote -- the provenance is that notebook's, not the assembler's.  Pair-difference "
            f"sigma over root two, mean over the four CFA planes, at offset "
            f"{src['bias_constants']['project_offset']['value']} and -10 C.  "
            f"{len(read_noise)} gains from {min(read_noise, key=float)} to "
            f"{max(read_noise, key=float)}, in tens with a grid of twos through the HCG "
            f"threshold.  Stops at {M.GAIN_MAX} per the gain-domain rule in CLAUDE.md; session "
            f"01's rows above it remain in bias_sweep.csv.  Uncertainty is the per-gain standard "
            f"error",
}

with MODEL_JSON.open("w") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {MODEL_JSON.name}: {len(constants)} constants")

## 6. The file read back through the gate it was written for

The assembler and the consumer are different code, so the file is loaded by `model.Constants` and
the model is run on it end to end. A constants file that its own consumer refuses is worse than no
file, and the only way to know is to try.

The run below is the first time this project's model has been evaluated on published constants
alone. What it prints is not a result - it is a demonstration that the pieces connect - and the
number it turns on is `efficiency`, which needs no object flux and no stack size and is therefore
the part that can be shown before contract 3 lands.

In [ ]:
c = M.Constants.load(MODEL_JSON)

g = M.g_at(LADDER_GAIN, c["system_gain"])
read_e = M.read_noise_e(LADDER_GAIN, c["read_noise_counts"], c["system_gain"])
f_sky = c.pick("F_sky", "G1")
kw = {"f_sky": f_sky, "dark": c["dark_current_bound"], "read_e": read_e,
      "t_dead": c["t_dead"]}

print(f"at gain {LADDER_GAIN}:  g = {g:.4f} e-/count,  R = {read_e:.3f} e-,  "
      f"F_sky(G1) = {f_sky:.3f} e-/px/s,  t_dead = {kw['t_dead']:.2f} s\n")
print("   t    exposing   sky-dominated   efficiency")
for t in (15, 30, 60, 120, 240, 480):
    a = kw["f_sky"] + kw["dark"]
    print(f"{t:5d}    {t / (t + kw['t_dead']):.4f}       {a * t / (a * t + read_e ** 2):.4f}"
          f"         {M.efficiency(t, **kw):.4f}")
for target in (0.90, 0.95, 0.99):
    print(f"  {target:.0%} of the asymptote at t = {M.t_for_efficiency(target, **kw):.1f} s")

### What the gate refuses, which is the other half of the check

Three refusals, each one a way a number could have reached a recommendation without anyone
deciding it should. They are asserted rather than printed because a gate that is documented and
not exercised is a gate that has already failed.

In [ ]:
for name, why in (("linear_to_at_ladder_gain", "the null this notebook wrote"),
                  ("ceiling", "absent: it lives in linearity_constants.json, published null")):
    try:
        c[name]
    except (ValueError, KeyError) as e:
        print(f"refused {name:26s} ({why})\n    {str(e)[:110]}")
    else:
        raise AssertionError(f"{name} should not have loaded")

try:
    M.g_at(600, c["system_gain"])
except ValueError as e:
    print(f"refused gain 600                    (outside the characterised domain)\n    {e}")

## 7. What this licenses, and what it does not

**Licensed.** The model runs on published constants with provenance, at any gain in 0-450, for
every term except the star-colour ceiling above gain 200. The efficiency curve above is a
prediction this project can defend line by line back to the frames.

**Not licensed.** Any statement about which exposure *wins*. That needs `eta_comb` on registered
lights, and the one in this file is a bias-stack upper bound that falls to 0.536 by N=128 - a
number about fixed pattern in bias frames, not about combining sky-limited subs. A six-hour night
of 15 s subs is N=464, past the end of the measured ladder, and `model.eta_at` refuses to
extrapolate there.

**Not touched.** No constant moved. Every value in `model_constants.json` is byte-for-byte what
its source file holds, apart from `read_noise_counts`, which is assembled from a sweep this
project published, and the two stanzas this notebook wrote itself and signed as its own.

**The open record decision.** `g` at gains 300 and 450 differs by 1.16% and 1.36% between the PTC
fit and session 04's restatement. This file carries the PTC table and the size of the gap; which
of the two is the record is a decision for `DECISIONS.md`, not for an assembler.